<a href="https://colab.research.google.com/github/theimhtikesoe/ai-rap-youtube-bot/blob/applio-training/applio/Applio_Training_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Applio Voice Model Training on Google Colab

This notebook prepares the official Applio environment, mounts Google Drive, copies the vocal dataset into Applio's expected dataset directory, and launches Applio. Training is then completed in Applio's Train tab.

**Use only voice recordings for which you have permission.** Keep the Colab runtime on GPU and save checkpoints to Drive because Colab sessions can disconnect.

In [1]:
# 1) Check the Colab runtime GPU
# Colab UI reports the selected GPU; some current runtimes do not expose nvidia-smi.
import os, shutil, subprocess
nvidia = shutil.which('nvidia-smi')
if nvidia:
    subprocess.run([nvidia], check=False)
else:
    print('nvidia-smi is unavailable in this Colab runtime; continuing because GPU runtime is selected in the UI.')
print('GPU check completed.')

GPU check completed.


In [2]:
# 2) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/Applio_Training'
SOURCE_DATASET = os.path.join(DRIVE_ROOT, 'dataset')
OUTPUT_MODELS = os.path.join(DRIVE_ROOT, 'output_models')
os.makedirs(OUTPUT_MODELS, exist_ok=True)
assert os.path.isdir(SOURCE_DATASET), f'Dataset folder not found: {SOURCE_DATASET}'
print('Dataset:', SOURCE_DATASET)
print('Model output:', OUTPUT_MODELS)

Mounted at /content/drive
Dataset: /content/drive/MyDrive/Applio_Training/dataset
Model output: /content/drive/MyDrive/Applio_Training/output_models


In [3]:
# 3) Clone the official Applio repository
%cd /content
!rm -rf Applio
!git clone --depth 1 https://github.com/iahispano/Applio.git
%cd /content/Applio
!bash run-install.sh

/content
Cloning into 'Applio'...
remote: Enumerating objects: 278, done.
remote: Counting objects: 100% (278/278), done.
remote: Compressing objects: 100% (234/234), done.
remote: Total 278 (delta 32), reused 111 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (278/278), 18.14 MiB | 19.79 MiB/s, done.
Resolving deltas: 100% (32/32), done.
/content/Applio
]0;Installer2026-09-25 05:51:00 - Attempting to install build tools...
2026-09-25 05:51:00 - Installing build-essential using apt...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Get:5 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [359 B]
Hit:7 http://

In [4]:
# 4) Copy and normalize the Drive dataset
# Applio expects a model-specific folder under assets/datasets.
import shutil, subprocess, pathlib
MODEL_NAME = 'Custom_Voice_Model'
APPLIO_DATASET = f'/content/Applio/assets/datasets/{MODEL_NAME}'
shutil.rmtree(APPLIO_DATASET, ignore_errors=True)
os.makedirs(APPLIO_DATASET, exist_ok=True)

source_files = []
for path in pathlib.Path(SOURCE_DATASET).rglob('*'):
    if path.is_file() and path.suffix.lower() in {'.wav', '.flac', '.m4a', '.mp3', '.ogg'}:
        source_files.append(path)
assert source_files, f'No audio files found in {SOURCE_DATASET}'

converted = 0
for source in sorted(source_files):
    target = pathlib.Path(APPLIO_DATASET) / (source.stem + '.wav')
    if source.suffix.lower() == '.wav':
        shutil.copy2(source, target)
    else:
        subprocess.run([
            'ffmpeg', '-y', '-i', str(source), '-ac', '1', '-ar', '40000',
            '-sample_fmt', 's16', str(target)
        ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        converted += 1

audio_files = sorted(pathlib.Path(APPLIO_DATASET).glob('*.wav'))
print(f'Prepared {len(audio_files)} WAV files in {APPLIO_DATASET}')
print(f'Converted compressed files: {converted}')
for path in audio_files:
    print(f' - {path.name}: {path.stat().st_size / 1024 / 1024:.1f} MB')

Prepared 18 WAV files in /content/Applio/assets/datasets/Custom_Voice_Model
Converted compressed files: 11
 - Fake_smile_vox.wav: 38.3 MB
 - Never_really_know vox.wav: 31.5 MB
 - Oh_vox.wav: 30.8 MB
 - Rhyzoe Vocal 2 .wav: 13.9 MB
 - Rhyzoe Vocal 5.wav: 19.1 MB
 - Rhyzoe vocal 1 .wav: 14.5 MB
 - Rhyzoe vocal 10.wav: 14.8 MB
 - Rhyzoe vocal 3.wav: 9.8 MB
 - Rhyzoe vocal 4.wav: 14.3 MB
 - Rhyzoe vocal 7.wav: 16.1 MB
 - Rhyzoe vocal 8.wav: 14.5 MB
 - Rhyzoe vocal 9.wav: 15.2 MB
 - Vocal 10mins.wav: 138.3 MB
 - effect vox long.wav: 135.8 MB
 - ego vox.wav: 27.4 MB
 - rhyzoe vocal 6.wav: 16.3 MB
 - vocal 10mins 2.wav: 150.4 MB
 - weak vocal 1.wav: 15.1 MB


In [5]:
# 5) Basic audio duration check
import wave
total_seconds = 0.0
for path in audio_files:
    try:
        with wave.open(str(path), 'rb') as audio:
            duration = audio.getnframes() / audio.getframerate()
            total_seconds += duration
            print(f'{path.name}: {duration / 60:.2f} min, {audio.getframerate()} Hz, {audio.getnchannels()} channel(s)')
    except wave.Error as error:
        raise RuntimeError(f'Unreadable WAV file: {path}: {error}')
print(f'Total readable WAV duration: {total_seconds / 60:.2f} minutes')
if total_seconds < 10 * 60:
    print('WARNING: official guidance recommends approximately 10-30 minutes of clean audio.')

Fake_smile_vox.wav: 3.79 min, 44100 Hz, 2 channel(s)
Never_really_know vox.wav: 3.12 min, 44100 Hz, 2 channel(s)
Oh_vox.wav: 3.05 min, 44100 Hz, 2 channel(s)
Rhyzoe Vocal 2 .wav: 3.03 min, 40000 Hz, 1 channel(s)
Rhyzoe Vocal 5.wav: 4.18 min, 40000 Hz, 1 channel(s)
Rhyzoe vocal 1 .wav: 3.17 min, 40000 Hz, 1 channel(s)
Rhyzoe vocal 10.wav: 3.24 min, 40000 Hz, 1 channel(s)
Rhyzoe vocal 3.wav: 2.13 min, 40000 Hz, 1 channel(s)
Rhyzoe vocal 4.wav: 3.12 min, 40000 Hz, 1 channel(s)
Rhyzoe vocal 7.wav: 3.51 min, 40000 Hz, 1 channel(s)
Rhyzoe vocal 8.wav: 3.18 min, 40000 Hz, 1 channel(s)
Rhyzoe vocal 9.wav: 3.32 min, 40000 Hz, 1 channel(s)
Vocal 10mins.wav: 13.70 min, 44100 Hz, 2 channel(s)
effect vox long.wav: 13.45 min, 44100 Hz, 2 channel(s)
ego vox.wav: 2.71 min, 44100 Hz, 2 channel(s)
rhyzoe vocal 6.wav: 3.56 min, 40000 Hz, 1 channel(s)
vocal 10mins 2.wav: 14.90 min, 44100 Hz, 2 channel(s)
weak vocal 1.wav: 3.30 min, 40000 Hz, 1 channel(s)
Total readable WAV duration: 90.49 minutes


## 6) Launch Applio

Run the next cell. When the public Gradio URL appears, open it. In the **Train** tab use:

- Model name: `Custom_Voice_Model`
- Dataset folder: the prepared model folder under `assets/datasets`
- Sample rate: choose `40k` or `48k` consistently with the selected pre-trained model
- Pitch extraction: `RMVPE`
- Save every epoch: `25`
- Total epochs: start around `300`, then monitor TensorBoard/loss and stop or resume based on quality
- Batch size: start conservatively; halve it if CUDA out-of-memory occurs

After training, use **Train Index**, then **Export Model**. Export the matching `.pth` and `.index` files into the Drive `output_models` folder.

In [6]:
# 7) Launch the official Applio UI
%cd /content/Applio
!python app.py --share

/content/Applio
Config file not found. Creating fresh from template.
Traceback (most recent call last):
  File "/content/Applio/app.py", line 110, in <module>
    from tabs.realtime.realtime import realtime_tab
  File "/content/Applio/tabs/realtime/realtime.py", line 2, in <module>
    import sounddevice as sd
  File "/usr/local/lib/python3.13/dist-packages/sounddevice.py", line 72, in <module>
    raise OSError('PortAudio library not found')
OSError: PortAudio library not found


In [7]:
# 8) Optional: verify exported model files after training
import glob, os
pth_files = glob.glob(os.path.join(OUTPUT_MODELS, '*.pth'))
index_files = glob.glob(os.path.join(OUTPUT_MODELS, '*.index'))
print('PTH files:')
for path in pth_files: print(' -', os.path.basename(path), os.path.getsize(path))
print('Index files:')
for path in index_files: print(' -', os.path.basename(path), os.path.getsize(path))
assert pth_files and index_files, 'Export both a .pth and a matching .index file before finishing.'

PTH files:
Index files:


AssertionError: Export both a .pth and a matching .index file before finishing.